# 05 · Validate net irrigation requirements

Checks unique keys, area and water units, numerical domain, and the expected 29,338 country–product–year–scenario states.

This notebook contains the scientific Python logic directly, not only a call to a .py script. Run cells in order after placing approved inputs under `data/`. All outputs are written to new directories under `runs/`; source files are never overwritten. For scientific scope and frozen evidence dependencies, see `docs/reproducibility.md`.

## Scientific methods and checks

In [ ]:
#!/usr/bin/env python3
"""QC the accepted net crop-requirement master (scientific route selection is upstream)."""
from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd

KEYS = ['country_iso3','fable_product','year','pathway']
REQUIRED = KEYS + ['water_requirement_mm_yr','water_requirement_m3_ha_yr','SPAM_irrigated_area_ha']


def validate(df: pd.DataFrame, expected_rows: int | None = 29338) -> dict:
    missing = sorted(set(REQUIRED)-set(df.columns))
    if missing: raise ValueError(f'Missing net master fields: {missing}')
    if df[KEYS].isna().any().any() or df.duplicated(KEYS).any():
        raise ValueError('Null or duplicated net master keys')
    v = df[['water_requirement_mm_yr','water_requirement_m3_ha_yr','SPAM_irrigated_area_ha']].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(v.to_numpy(dtype=float)).all() or (v < 0).any().any():
        raise ValueError('Net master has missing, negative or non-finite scientific values')
    if not np.allclose(v['water_requirement_m3_ha_yr'],v['water_requirement_mm_yr']*10,atol=1e-7,rtol=1e-10):
        raise ValueError('mm/year to m³/ha/year conversion differs from ×10')
    report = {'rows':len(df),'countries':int(df.country_iso3.nunique()),'products':int(df.fable_product.nunique()),
              'years':sorted(int(i) for i in df.year.unique()),'pathways':sorted(df.pathway.astype(str).unique().tolist()),
              'expected_rows':expected_rows,'expected_rows_match': expected_rows is None or len(df)==expected_rows}
    if expected_rows is not None and len(df)!=expected_rows:
        raise ValueError(f'Expected {expected_rows:,} net states, found {len(df):,}')
    return report


## Inputs and output location

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
NET_FILE = DATA / 'crop' / '01_FABLE_crop_water_requirement_master_2000_2020_FINAL.csv.gz'


## Execute and inspect

In [ ]:
if not NET_FILE.is_file():
    raise FileNotFoundError(NET_FILE)
net_report = validate(pd.read_csv(NET_FILE, low_memory=False), expected_rows=29338)
print(json.dumps(net_report, indent=2))
print('STATUS: PASS_ACCEPTED_NET_MASTER_NUMERIC_QC')
